# Task 8 – Data Lineage Tracker

**RecoMart Data Management Pipeline**

Tracks the full provenance of every data asset across all pipeline stages:

## Task Structure
1. **Task 1**: Raw data generation (batch and streaming sources)
2. **Task 2**: Streaming ingestion (Auto Loader, checkpointing)
3. **Task 3**: Batch ingestion (Delta Lake tables)
4. **Task 4**: Validation, preparation, and EDA
5. **Task 5-6**: Feature engineering (user, item, user-item features)
6. **Task 7**: Feature store (online/offline, training datasets)

## Lineage Tracking
For each asset, we track:
- **Source**: Which stage produced it
- **Dependencies**: Which inputs it depends on
- **Transformations**: What operations were applied
- **Metadata**: MD5 checksum, file size, timestamp, version

## Outputs
- `lineage/lineage_graph.json` – Full DAG of all assets
- `lineage/lineage_report.txt` – Human-readable lineage report

In [0]:
import json
import hashlib
import os
import subprocess
from datetime import datetime

def md5(path):
    """Calculate MD5 checksum of a file."""
    if not os.path.exists(path):
        return "FILE_NOT_FOUND"
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def file_size_kb(path):
    """Get file size in KB."""
    if not os.path.exists(path):
        return 0
    return round(os.path.getsize(path) / 1024, 1)

def git_hash():
    """Get current Git commit hash (short form)."""
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"],
            stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception:
        return "uncommitted"

print("Utility functions loaded successfully")

In [0]:
# Base paths for Databricks workspace
BASE = "/Workspace/Users/2025ae05764@wilp.bits-pilani.ac.in/RecomartEcommerce/2_medallion_processing"
OUT_DIR = f"{BASE}/lineage"

# Create output directory if it doesn't exist
os.makedirs(OUT_DIR, exist_ok=True)

# Capture current timestamp and git hash
NOW = datetime.now().isoformat()
GIT_COMMIT = git_hash()

print(f"Base directory: {BASE}")
print(f"Output directory: {OUT_DIR}")
print(f"Timestamp: {NOW}")
print(f"Git commit: {GIT_COMMIT}")

In [0]:
def build_lineage_graph():
    """
    Build complete lineage graph tracking all data assets across pipeline stages.
    Returns dict with nodes (assets) and edges (dependencies).
    """
    nodes = []
    edges = []

    def node(asset_id, stage, task, path, description,
             asset_type="data", transformations=None):
        """Register a data asset node."""
        full = f"{BASE}/{path}"
        nodes.append({
            "id":            asset_id,
            "stage":         stage,
            "task":          task,
            "path":          path,
            "description":   description,
            "asset_type":    asset_type,  # data | model | metrics | report
            "md5":           md5(full),
            "size_kb":       file_size_kb(full),
            "transformations": transformations or [],
            "git_commit":    GIT_COMMIT,
            "created_at":    NOW,
            "version":       "v1.0",
        })

    def edge(src, tgt, operation):
        """Register a lineage edge (dependency relationship)."""
        edges.append({"from": src, "to": tgt, "operation": operation})

    # ══════════════════════════════════════════════════════════════════════════
    # TASK 1: RAW DATA GENERATION
    # ══════════════════════════════════════════════════════════════════════════
    node("raw_clickstream_batch", "generation", "Task 1",
         "raw_data/batch/user_interaction_logs.csv",
         "5,000 raw clickstream events for batch ingestion",
         transformations=["Generated synthetically with random.seed(42)"])

    node("raw_transactions_batch", "generation", "Task 1",
         "raw_data/batch/purchase_history.csv",
         "1,500 raw purchase records for batch ingestion",
         transformations=["Generated synthetically with random.seed(42)"])

    node("raw_catalog_batch", "generation", "Task 1",
         "raw_data/batch/product_catalog.json",
         "500 product items from simulated REST API",
         transformations=["Simulated paginated REST API response"])

    node("raw_external_batch", "generation", "Task 1",
         "raw_data/batch/external_signals.json",
         "500 items with popularity, sentiment, trend scores",
         transformations=["Simulated third-party API call"])

    node("raw_clickstream_stream", "generation", "Task 1",
         "raw_data/streaming/user_interactions/",
         "Streaming clickstream events (10 batches, 50 events each)",
         transformations=["Generated for Auto Loader streaming ingestion"])

    # ══════════════════════════════════════════════════════════════════════════
    # TASK 2: STREAMING INGESTION
    # ══════════════════════════════════════════════════════════════════════════
    node("streaming_bronze", "streaming_ingest", "Task 2",
         "_checkpoints/streaming_bronze/",
         "Auto Loader streaming checkpoint - incremental event ingestion",
         asset_type="checkpoint",
         transformations=[
             "Auto Loader with cloudFiles.format=csv",
             "Incremental ingestion with checkpointing",
             "Schema inference and evolution",
         ])

    node("streaming_delta", "streaming_ingest", "Task 2",
         "delta_tables/streaming_events/",
         "Delta table with streaming clickstream events",
         transformations=[
             "writeStream.format('delta')",
             "APPEND mode with Auto Loader",
             "Processed 500 events across 10 micro-batches",
         ])

    edge("raw_clickstream_stream", "streaming_bronze", "ingested_by")
    edge("streaming_bronze", "streaming_delta", "checkpointed_to")

    # ══════════════════════════════════════════════════════════════════════════
    # TASK 3: BATCH INGESTION
    # ══════════════════════════════════════════════════════════════════════════
    node("bronze_clickstream", "batch_ingest", "Task 3",
         "delta_tables/bronze_clickstream/",
         "Bronze layer: Raw clickstream events in Delta format",
         transformations=[
             "spark.read.csv() with schema inference",
             "write.format('delta').mode('overwrite')",
             "5,000 events ingested",
         ])

    node("bronze_transactions", "batch_ingest", "Task 3",
         "delta_tables/bronze_transactions/",
         "Bronze layer: Raw purchase transactions in Delta format",
         transformations=[
             "spark.read.csv() with schema inference",
             "write.format('delta').mode('overwrite')",
             "1,500 transactions ingested",
         ])

    node("bronze_catalog", "batch_ingest", "Task 3",
         "delta_tables/bronze_catalog/",
         "Bronze layer: Raw product catalog in Delta format",
         transformations=[
             "spark.read.json() from API simulation",
             "write.format('delta').mode('overwrite')",
             "500 products ingested",
         ])

    node("bronze_external", "batch_ingest", "Task 3",
         "delta_tables/bronze_external/",
         "Bronze layer: External signals in Delta format",
         transformations=[
             "spark.read.json() from external API",
             "write.format('delta').mode('overwrite')",
             "500 signal records ingested",
         ])

    edge("raw_clickstream_batch", "bronze_clickstream", "ingested_into")
    edge("raw_transactions_batch", "bronze_transactions", "ingested_into")
    edge("raw_catalog_batch", "bronze_catalog", "ingested_into")
    edge("raw_external_batch", "bronze_external", "ingested_into")

    # ══════════════════════════════════════════════════════════════════════════
    # TASK 4: VALIDATION, PREPARATION, EDA
    # ══════════════════════════════════════════════════════════════════════════
    node("dq_report", "validation", "Task 4",
         "validation/data_quality_report.json",
         "Data quality checks: missing, duplicates, domain, range, referential integrity",
         asset_type="report",
         transformations=[
             "Missing value analysis per column",
             "Duplicate key detection (event_id, order_id, item_id)",
             "Domain checks: event_type ∈ {view,click,add_to_cart,purchase}",
             "Range checks: rating∈[1,5], price>0, popularity∈[0,1]",
             "Cross-source referential integrity",
         ])

    node("eda_report", "eda", "Task 4",
         "eda/exploratory_analysis.html",
         "Exploratory data analysis with visualizations",
         asset_type="report",
         transformations=[
             "Distribution analysis of key metrics",
             "Temporal patterns in user behavior",
             "Correlation analysis between features",
             "Cohort analysis and segmentation",
         ])

    node("silver_clickstream", "preparation", "Task 4",
         "delta_tables/silver_clickstream/",
         "Silver layer: Cleaned and validated clickstream events",
         transformations=[
             "fillna(city, 'Unknown')",
             "cast(timestamp, TimestampType)",
             "drop_duplicates()",
             "filter event_type ∈ valid set",
             "add hour, day_of_week columns",
         ])

    node("silver_transactions", "preparation", "Task 4",
         "delta_tables/silver_transactions/",
         "Silver layer: Cleaned and validated transactions",
         transformations=[
             "fillna(payment_method, 'UNSPECIFIED')",
             "drop_duplicates(order_id, keep='last')",
             "filter unit_price > 0 and quantity > 0",
             "total_price = unit_price × quantity (recalculated)",
             "Removed 44 duplicate order_ids",
         ])

    node("silver_catalog", "preparation", "Task 4",
         "delta_tables/silver_catalog/",
         "Silver layer: Active products with validated ratings",
         transformations=[
             "filter is_active == True (removed 25 inactive)",
             "rating.clip(1.0, 5.0)",
             "drop_duplicates(item_id)",
             "475 active products",
         ])

    node("silver_external", "preparation", "Task 4",
         "delta_tables/silver_external/",
         "Silver layer: External signals with clipped scores",
         transformations=[
             "popularity_score.clip(0, 1)",
             "avg_sentiment.clip(-1, 1)",
             "return_rate.clip(0, 1)",
             "drop_duplicates(item_id)",
         ])

    edge("bronze_clickstream", "dq_report", "profiled_by")
    edge("bronze_transactions", "dq_report", "profiled_by")
    edge("bronze_catalog", "dq_report", "profiled_by")
    edge("bronze_external", "dq_report", "profiled_by")
    edge("dq_report", "silver_clickstream", "guided_by")
    edge("dq_report", "silver_transactions", "guided_by")
    edge("dq_report", "silver_catalog", "guided_by")
    edge("dq_report", "silver_external", "guided_by")
    edge("bronze_clickstream", "silver_clickstream", "cleaned_into")
    edge("bronze_transactions", "silver_transactions", "cleaned_into")
    edge("bronze_catalog", "silver_catalog", "cleaned_into")
    edge("bronze_external", "silver_external", "cleaned_into")
    edge("silver_clickstream", "eda_report", "analyzed_in")
    edge("silver_transactions", "eda_report", "analyzed_in")
    edge("silver_catalog", "eda_report", "analyzed_in")

    # ══════════════════════════════════════════════════════════════════════════
    # TASK 5-6: FEATURE ENGINEERING
    # ══════════════════════════════════════════════════════════════════════════
    node("user_features", "feature_engineer", "Task 5-6",
         "features/user_features.csv",
         "200 users × 22 features: engagement, spend, recency, preferences",
         transformations=[
             "Event count aggregations per user",
             "Spend statistics from transactions",
             "Distinct items and categories",
             "Recency and tenure calculations",
             "Preferred category/device (label encoded)",
             "Activity score normalization",
         ])

    node("item_features", "feature_engineer", "Task 5-6",
         "features/item_features.csv",
         "475 items × 30 features: interactions, conversion, quality, popularity",
         transformations=[
             "Interaction counts from clickstream",
             "Conversion rate = purchases / interactions",
             "Average rating and review counts",
             "Popularity and sentiment scores",
             "Price log-transformation",
             "Stock normalization",
             "Category and brand encoding",
             "Trend score and return rate",
         ])

    node("user_item_features", "feature_engineer", "Task 5-6",
         "features/user_item_features.csv",
         "4,650 user-item pairs × 10 features: interaction scores, purchase labels",
         transformations=[
             "Interaction score = weighted sum(events)",
             "Purchase flag from transactions join",
             "Event type counts (views, clicks, carts, purchases)",
             "Interaction score normalization",
         ])

    node("co_purchase_pairs", "feature_engineer", "Task 5-6",
         "features/co_purchase_pairs.csv",
         "0 co-purchase pairs (insufficient overlap in sample data)",
         transformations=[
             "groupby(user_id)[item_id].apply(list)",
             "combinations(items, r=2) → item pairs",
             "count co-occurrences across all users",
         ])

    node("feature_metadata", "feature_engineer", "Task 5-6",
         "features/feature_metadata.csv",
         "57 feature definitions with types and descriptions",
         asset_type="metadata",
         transformations=[
             "Catalog all engineered features",
             "Document feature types (continuous, categorical, binary)",
             "Track feature groups (user, item, user_item)",
         ])

    edge("silver_clickstream", "user_features", "aggregated_into")
    edge("silver_transactions", "user_features", "joined_into")
    edge("silver_catalog", "user_features", "joined_into")
    edge("silver_clickstream", "item_features", "aggregated_into")
    edge("silver_catalog", "item_features", "joined_into")
    edge("silver_external", "item_features", "joined_into")
    edge("silver_clickstream", "user_item_features", "aggregated_into")
    edge("silver_transactions", "user_item_features", "joined_into")
    edge("silver_transactions", "co_purchase_pairs", "derived_from")

    # ══════════════════════════════════════════════════════════════════════════
    # TASK 7: FEATURE STORE
    # ══════════════════════════════════════════════════════════════════════════
    node("feature_registry", "feature_store", "Task 7",
         "feature_store/feature_registry.db",
         "SQLite registry: 57 features across 3 groups (user, item, user_item)",
         asset_type="metadata",
         transformations=[
             "FeatureRegistry with feature_definitions table",
             "Feature versioning with checksums",
             "Feature set definitions",
             "9 materialization snapshots tracked",
         ])

    node("offline_store", "feature_store", "Task 7",
         "feature_store/offline_store.db",
         "SQLite offline store: Historical features for training",
         transformations=[
             "Write user_features (200 rows)",
             "Write item_features (475 rows)",
             "Write user_item_features (4,650 rows)",
             "Point-in-time correct joins",
         ])

    node("online_store", "feature_store", "Task 7",
         "feature_store/online_store_cache.json",
         "In-memory cache: 675 keys (200 users + 475 items) with TTL=3600s",
         transformations=[
             "Materialize from offline store",
             "Key-value store for low-latency serving",
             "100% cache hit rate in demo",
         ])

    node("training_dataset", "feature_store", "Task 7",
         "feature_store/training_dataset.csv",
         "500 rows × 28 columns — point-in-time join for model training",
         transformations=[
             "OfflineStore.get_training_dataset()",
             "JOIN user_item_features ON (user_id, item_id)",
             "LEFT JOIN user_features ON user_id",
             "LEFT JOIN item_features ON item_id",
             "Label = ui_has_purchased (binary)",
             "Label distribution: {0: 490, 1: 10}",
         ])

    edge("user_features", "feature_registry", "registered_in")
    edge("item_features", "feature_registry", "registered_in")
    edge("user_item_features", "feature_registry", "registered_in")
    edge("feature_metadata", "feature_registry", "cataloged_in")
    edge("user_features", "offline_store", "written_to")
    edge("item_features", "offline_store", "written_to")
    edge("user_item_features", "offline_store", "written_to")
    edge("offline_store", "online_store", "materialized_into")
    edge("offline_store", "training_dataset", "joined_into")
    edge("user_features", "training_dataset", "joined_into")
    edge("item_features", "training_dataset", "joined_into")
    edge("user_item_features", "training_dataset", "joined_into")

    return {
        "nodes": nodes,
        "edges": edges,
        "generated_at": NOW,
        "git_commit": GIT_COMMIT,
        "total_nodes": len(nodes),
        "total_edges": len(edges)
    }

print("build_lineage_graph() function defined")

In [0]:
# Build the lineage graph
print("Building lineage graph ...")
graph = build_lineage_graph()

# Save as JSON
with open(f"{OUT_DIR}/lineage_graph.json", "w") as f:
    json.dump(graph, f, indent=2)

print(f"  Total nodes: {graph['total_nodes']}")
print(f"  Total edges: {graph['total_edges']}")
print(f"  Generated at: {graph['generated_at']}")
print(f"  Git commit: {graph['git_commit']}")
print(f"\nLineage graph saved to: {OUT_DIR}/lineage_graph.json")

In [0]:
# Generate human-readable lineage report
lines = []
lines.append("=" * 70)
lines.append("RECOMART — DATA LINEAGE REPORT")
lines.append(f"Generated : {NOW}")
lines.append(f"Git commit: {GIT_COMMIT}")
lines.append("=" * 70)

# Define stage order and labels
STAGE_ORDER = [
    "generation",
    "streaming_ingest",
    "batch_ingest",
    "validation",
    "preparation",
    "eda",
    "feature_engineer",
    "feature_store"
]

TASK_LABELS = {
    "generation":        "Task 1 — Data Generation (Batch & Streaming)",
    "streaming_ingest":  "Task 2 — Streaming Ingestion (Auto Loader)",
    "batch_ingest":      "Task 3 — Batch Ingestion (Delta Lake)",
    "validation":        "Task 4 — Data Validation",
    "preparation":       "Task 4 — Data Preparation",
    "eda":               "Task 4 — Exploratory Data Analysis",
    "feature_engineer":  "Task 5-6 — Feature Engineering",
    "feature_store":     "Task 7 — Feature Store",
}

# Iterate through stages and generate report
for stage in STAGE_ORDER:
    stage_nodes = [n for n in graph["nodes"] if n["stage"] == stage]
    if not stage_nodes:
        continue
    
    lines.append(f"\n{'─'*70}")
    lines.append(f"  {TASK_LABELS.get(stage, stage)}")
    lines.append(f"{'─'*70}")
    
    for n in stage_nodes:
        lines.append(f"\n  ASSET : {n['id']}")
        lines.append(f"  Path  : {n['path']}")
        lines.append(f"  Size  : {n['size_kb']} KB")
        lines.append(f"  MD5   : {n['md5']}")
        lines.append(f"  Desc  : {n['description']}")

        # Upstream dependencies
        upstream = [e["from"] for e in graph["edges"] if e["to"] == n["id"]]
        if upstream:
            lines.append(f"  From  : {', '.join(upstream)}")

        # Transformations
        if n["transformations"]:
            lines.append("  Transforms:")
            for t in n["transformations"]:
                lines.append(f"    → {t}")

        # Downstream dependents
        downstream = [e["to"] for e in graph["edges"] if e["from"] == n["id"]]
        if downstream:
            lines.append(f"  Used by: {', '.join(downstream)}")

lines.append(f"\n{'=' * 70}")
lines.append(f"Total assets tracked : {graph['total_nodes']}")
lines.append(f"Total lineage edges  : {graph['total_edges']}")
lines.append("=" * 70)

# Save report
report_text = "\n".join(lines)
with open(f"{OUT_DIR}/lineage_report.txt", "w") as f:
    f.write(report_text)

# Print preview
print("\nLineage report preview:")
print("=" * 70)
for ln in lines[:50]:
    print(ln)
if len(lines) > 50:
    print("  ...")
    print(f"  ({len(lines) - 50} more lines)")
print(f"\nFull report saved to: {OUT_DIR}/lineage_report.txt")